# Interpolate between two orthogonal matrices

Interpolate between two orthogonal matrices so that each interpolant is orthogonal (geodesic on the orthogonal group). Then use three of these matrices as FDN feedback matrices and plot their impulse responses via `pyFDN.dss2impz`.

Reference: *Schlecht, S., Habets, E. (2015). Practical considerations of time-varying feedback delay networks.* Proc. Audio Eng. Soc. Conv.

- Original version: Sebastian J. Schlecht, Friday, 10. April 2020
- Translation: Sebastian J. Schlecht, Thursday, 19. February 2026

In [ ]:
import math
import numpy as np
import plotly.graph_objects as go
from scipy.linalg import hadamard
import pyFDN

## Parameters and interpolation

Start from a Hadamard matrix (sign-normalized) and the identity; interpolate along 20 steps and verify each interpolant is orthogonal.

In [ ]:
N = 4
# Hadamard, normalized and sign-normalized on diagonal (match MATLAB fdnMatrixGallery)
A = hadamard(N) / math.sqrt(N)
A = A @ np.diag(np.sign(np.diag(A)))
B = np.eye(N)

num_t = 20
T = np.linspace(0, 1, num_t)
C = np.zeros((N, N, num_t))
for it, t in enumerate(T):
    C[:, :, it] = pyFDN.interpolate_orthogonal(A, B, t)

is_orth = [pyFDN.is_orthogonal(C[:, :, it]) for it in range(num_t)]
assert all(is_orth), "All interpolants should be orthogonal"
print("All interpolants are orthogonal.")

## Animation of the interpolated feedback matrix

Animate **C(t)** as a heatmap; use the slider or play button to move along the geodesic from A (t=0) to B (t=1).

In [ ]:
# Plotly Express: one call, slider + play (animation_frame = axis to slice)
import plotly.express as px

zmin = -1
zmax = 1
fig = px.imshow(
    C,
    animation_frame=2,
    origin="upper",
    range_color=(zmin, zmax),
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    labels=dict(animation_frame="t"),
)
fig.update_layout(height=420)
# Optional: label slider with T values
fig.layout.sliders[0].currentvalue = dict(prefix="t = ", visible=True)
for k, step in enumerate(fig.layout.sliders[0].steps):
    step["label"] = f"{T[k]:.2f}"
fig.show()

## Three impulse responses via dss2impz

Use three feedback matrices (t=0, t=0.63, t=1) in a minimal FDN and compute the impulse response for each with `pyFDN.dss2impz`. Plot the three IRs (mu-law encoded for visibility).

In [ ]:
ir_len = 2000
delays = np.array([101, 163, 197, 241], dtype=np.int64)
g = 0.999
b = np.ones((N, 1))
c = np.ones((1, N))
d = np.array([[0.0]])

# Pick three interpolants: start (A), middle, end (B)
idx = [0, num_t // 3 * 2, num_t - 1]
labels = [f"t={T[i]:.2f}" for i in idx]
irs = []
for i in idx:
    Af = C[:, :, i] @ np.diag(g ** delays)  # feedback matrix with gain per delay
    ir = pyFDN.dss_to_impz(ir_len, delays, Af, b, c, d)
    ir = np.asarray(ir).squeeze().ravel()
    irs.append(ir)

fig = go.Figure()
for i, (ir, label) in enumerate(zip(irs, labels)):
    fig.add_trace(go.Scatter(y=pyFDN.mulaw_encode(ir) + 2*i, mode="lines", name=label, line=dict(width=1.5)))
fig.update_layout(
    title="FDN impulse response for three interpolated feedback matrices",
    xaxis_title="Time [samples]",
    yaxis_title="Amplitude [μ-law]",
    legend_title="Interpolation",
    hovermode="x unified",
    template="plotly_white",
    height=400,
)
fig.show()